## Structured Output

--> Models can be requested to provide their response in a formate matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. Langchain support multiple schema types and methods for enforcing structured output 

## Pydantic

--> pydantic models provide the richest feature set with field validation, descriptions, and nested structures. 

In [8]:
import os 
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY') #type:ignore
model = init_chat_model("groq:openai/gpt-oss-120b")

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str =Field(description="The title of the movie")
    year : int =Field(description="This year the movie was released")
    director : str =Field(description="The director of the movie")
    rating : float =Field(description="The movies rating out of 10")

In [10]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.17'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000252811FE5D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000252811FEFD0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [20]:
model_with_structure.invoke("Provide details about the movie thor ragnarok")

Movie(title='Thor: Ragnarok', year=2017, director='Taika Waititi', rating=7.9)

In [21]:
response = model_with_structure.invoke("Provide details about the movie thor ragnarok")
response

Movie(title='Thor: Ragnarok', year=2017, director='Taika Waititi', rating=7.9)

## Message output alongside parsed structure



In [23]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    """ A movie with details. """
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")
    
model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response
    

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie Inception. Use the provided function to return a movie object with director, rating, title, year. So we need to call function with those fields. Inception: director Christopher Nolan, rating maybe 8.8 (IMDb). Year 2010. Title "Inception". Use function.', 'tool_calls': [{'id': 'fc_112cdacd-313a-4ec6-84a4-035e5a8f6b32', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 165, 'total_tokens': 286, 'completion_time': 0.253637489, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.038386142, 'prompt_tokens_details': None, 'queue_time': 0.292983276, 'total_time': 0.292023631}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_854fa9be4c', 'service_tier': 'on_demand', 'finish_

## Nested Structure

In [26]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
class Movie_Details(BaseModel):
    title:str
    year:int
    title:str
    cast : list[Actor]
    genres:list[str]
    budget : float | None = Field(None,description="budget in million USD")

model_with_structure = model.with_structured_output(Movie_Details)

response = model_with_structure.invoke("Provide details about the movie thor")
response

Movie_Details(title='Thor', year=2011, cast=[Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Tom Hiddleston', role='Loki'), Actor(name='Natalie Portman', role='Jane Foster'), Actor(name='Anthony Hopkins', role='Odin'), Actor(name='Stellan Skarsgård', role='Erik Selvig')], genres=['Action', 'Adventure', 'Fantasy', 'Science Fiction'], budget=150000000.0)